# Aula 06: Lógica de Predicados, Quantificadores e Módulo de Varredura Global de Sensores

**Disciplina:** ECAA08 — Automática (2026.2) — UNIFEI  
**Projeto:** SCADA-Core Automática / Planta de Fertilizantes Químicos  
**Área:** Engenharia de Controle e Automação & Matemática Discreta  

---

## 1. Contexto e Objetivos

Nas aulas anteriores modelamos regras de intertravamento utilizando **Lógica Proposicional**. Porém, plantas químicas industriais modernas possuem centenas de instrumentos distribuídos em diferentes setores. Tratar cada sensor como uma variável proposicional isolada torna o código de supervisão rígido e difícil de escalar.

Nesta aula, implementamos a **Lógica de Primeira Ordem (Lógica de Predicados)** para construir o **Módulo de Varredura Global de Sensores** (*SCADA State Scanning Engine*), permitindo avaliar sentenças universais ($\forall$) e existenciais ($\exists$) sobre o ecossistema de instrumentos da planta de fertilizantes.

In [ ]:
import time
import random
from enum import Enum
from dataclasses import dataclass, field
from typing import List, Dict, Callable, Tuple, Any, Optional

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

def exibir_tabela(dados, titulo=""):
    if titulo:
        print(f"\n=== {titulo} ===")
    if HAS_PANDAS:
        display(pd.DataFrame(dados)) if 'display' in globals() else print(pd.DataFrame(dados))
    else:
        if isinstance(dados, dict):
            for k, v in dados.items():
                print(f"{k:30}: {v}")
        elif isinstance(dados, list):
            for item in dados:
                print(item)

print("Ambiente de desenvolvimento da Aula 06 inicializado com sucesso.")

## 2. Modelagem das Entidades da Planta de Fertilizantes

Definimos os setores, tipos de instrumentos conforme a norma ISA-5.1 e a estrutura de dados de cada tag de instrumentação.

In [ ]:
class Setor(Enum):
    SETOR_100 = "Setor 100 - Reação e Neutralização (NH3 + H3PO4)"
    SETOR_200 = "Setor 200 - Granulação e Secagem NPK"
    SETOR_300 = "Setor 300 - Parque de Tancagem e Transferência"

class TipoInstrumento(Enum):
    TRANSMISSOR_PRESSAO = "PT"
    TRANSMISSOR_TEMPERATURA = "TT"
    DETECTOR_GAS = "AT"
    TRANSMISSOR_NIVEL = "LT"
    CHAVE_FLUXO = "FS"
    VALVULA_BLOQUEIO = "XV"
    BOMBA_MOTOR = "M_P"

@dataclass
class Instrumento:
    tag: str
    tipo: TipoInstrumento
    setor: Setor
    descricao: str
    online: bool = True
    calibrado: bool = True
    valor_atual: float = 0.0
    unidade: str = ""
    limite_critico_alto: Optional[float] = None
    limite_critico_baixo: Optional[float] = None
    fim_de_curso_aberto: Optional[bool] = None  # Para válvulas
    motor_ligado: Optional[bool] = None         # Para motores e bombas

def criar_parque_instrumentos() -> List[Instrumento]:
    return [
        # Setor 100: Reator de Neutralização
        Instrumento("PT-101", TipoInstrumento.TRANSMISSOR_PRESSAO, Setor.SETOR_100, "Pressão Reator R-101", valor_atual=140.0, unidade="bar", limite_critico_alto=180.0),
        Instrumento("TT-101", TipoInstrumento.TRANSMISSOR_TEMPERATURA, Setor.SETOR_100, "Temperatura Zona Reação", valor_atual=165.0, unidade="°C", limite_critico_alto=200.0),
        Instrumento("TT-102", TipoInstrumento.TRANSMISSOR_TEMPERATURA, Setor.SETOR_100, "Temperatura Topo Reator", valor_atual=150.0, unidade="°C", limite_critico_alto=190.0),
        Instrumento("AT-101", TipoInstrumento.DETECTOR_GAS, Setor.SETOR_100, "Detector Amônia NH3 Área Síntese", valor_atual=5.2, unidade="ppm", limite_critico_alto=25.0),
        Instrumento("LT-101", TipoInstrumento.TRANSMISSOR_NIVEL, Setor.SETOR_100, "Nível Reator Slurry", valor_atual=65.0, unidade="%", limite_critico_baixo=15.0, limite_critico_alto=90.0),
        Instrumento("XV-101", TipoInstrumento.VALVULA_BLOQUEIO, Setor.SETOR_100, "Válvula Alimentação NH3", fim_de_curso_aberto=True),
        Instrumento("XV-102", TipoInstrumento.VALVULA_BLOQUEIO, Setor.SETOR_100, "Válvula Alimentação H3PO4", fim_de_curso_aberto=True),
        Instrumento("P-101", TipoInstrumento.BOMBA_MOTOR, Setor.SETOR_100, "Bomba de Ácido Fosfórico", motor_ligado=True),

        # Setor 200: Granulação e Secagem
        Instrumento("FS-201", TipoInstrumento.CHAVE_FLUXO, Setor.SETOR_200, "Fluxo de Ar do Exaustor", valor_atual=1200.0, unidade="m3/h", limite_critico_baixo=500.0),
        Instrumento("TT-201", TipoInstrumento.TRANSMISSOR_TEMPERATURA, Setor.SETOR_200, "Temperatura Entrada Secador", valor_atual=115.0, unidade="°C", limite_critico_alto=150.0),
        Instrumento("PT-201", TipoInstrumento.TRANSMISSOR_PRESSAO, Setor.SETOR_200, "Pressão na Câmara do Granulador", valor_atual=1.2, unidade="bar", limite_critico_alto=3.0),
        Instrumento("XV-201", TipoInstrumento.VALVULA_BLOQUEIO, Setor.SETOR_200, "Válvula Injeção Slurry Quente", fim_de_curso_aberto=True),
        Instrumento("M-201", TipoInstrumento.BOMBA_MOTOR, Setor.SETOR_200, "Motor do Tambor Granulador", motor_ligado=True),

        # Setor 300: Parque de Tancagem
        Instrumento("AT-301", TipoInstrumento.DETECTOR_GAS, Setor.SETOR_300, "Detector Amônia Tanque Pulmão", valor_atual=3.1, unidade="ppm", limite_critico_alto=25.0),
        Instrumento("LT-301", TipoInstrumento.TRANSMISSOR_NIVEL, Setor.SETOR_300, "Nível Tanque Armazenagem NH3", valor_atual=78.0, unidade="%", limite_critico_alto=95.0),
        Instrumento("XV-301", TipoInstrumento.VALVULA_BLOQUEIO, Setor.SETOR_300, "Válvula Isolamento Linha de Transferência", fim_de_curso_aberto=True),
    ]

instrumentos_base = criar_parque_instrumentos()
print(f"Total de instrumentos cadastrados na planta: {len(instrumentos_base)}")

## 3. Motor Algorítmico de Quantificação e Predicados Lógicos

Na matemática discreta, os quantificadores $\forall$ e $\exists$ operam sobre um domínio $\mathcal{U}$:
- **Quantificador Universal ($\forall x, P(x)$):** Retorna `True` se e somente se todos os elementos satisfazem $P(x)$. A avaliação utiliza curto-circuito na primeira falha (contraexemplo).
- **Quantificador Existencial ($\exists x, Q(x)$):** Retorna `True` se ao menos um elemento satisfaz $Q(x)$. A avaliação utiliza curto-circuito no primeiro acerto (testemunha).

Para subconjuntos com predicado guardião $S(x)$:
- $\forall x \in S, P(x) \iff \forall x (S(x) \rightarrow P(x))$
- $\exists x \in S, Q(x) \iff \exists x (S(x) \land Q(x))$

In [ ]:
def forall(dominio: List[Any], predicado: Callable[[Any], bool]) -> Tuple[bool, List[Any]]:
    """
    Avalia formalmente o Quantificador Universal: forall x in D, P(x)
    Retorna (Verdadeiro/Falso, Lista de Contraexemplos que violam P(x))
    """
    contraexemplos = []
    for x in dominio:
        if not predicado(x):
            contraexemplos.append(x)
    return (len(contraexemplos) == 0, contraexemplos)

def exists(dominio: List[Any], predicado: Callable[[Any], bool]) -> Tuple[bool, List[Any]]:
    """
    Avalia formalmente o Quantificador Existencial: exists x in D, Q(x)
    Retorna (Verdadeiro/Falso, Lista de Testemunhas que satisfazem Q(x))
    """
    testemunhas = []
    for x in dominio:
        if predicado(x):
            testemunhas.append(x)
    return (len(testemunhas) > 0, testemunhas)

def forall_in(dominio: List[Any], guarda: Callable[[Any], bool], predicado: Callable[[Any], bool]) -> Tuple[bool, List[Any]]:
    """
    Quantificacao Universal Restrita: forall x in D, S(x) -> P(x)
    """
    subdominio = [x for x in dominio if guarda(x)]
    return forall(subdominio, predicado)

def exists_in(dominio: List[Any], guarda: Callable[[Any], bool], predicado: Callable[[Any], bool]) -> Tuple[bool, List[Any]]:
    """
    Quantificacao Existencial Restrita: exists x in D, S(x) and Q(x)
    """
    subdominio = [x for x in dominio if guarda(x)]
    return exists(subdominio, predicado)

print("Motor de quantificacao logica de 1a ordem carregado.")

## 4. Definição dos Predicados Operacionais da Fábrica

In [ ]:
# Predicados de Integridade Física do Instrumento
def is_online(inst: Instrumento) -> bool:
    return inst.online

def is_calibrado(inst: Instrumento) -> bool:
    return inst.calibrado

def is_saudavel(inst: Instrumento) -> bool:
    return is_online(inst) and is_calibrado(inst)

# Predicados de Processo e Segurança Crítica
def is_vazamento_gas(inst: Instrumento) -> bool:
    if inst.tipo == TipoInstrumento.DETECTOR_GAS and inst.limite_critico_alto is not None:
        return inst.valor_atual > inst.limite_critico_alto
    return False

def is_sobrepressao(inst: Instrumento) -> bool:
    if inst.tipo == TipoInstrumento.TRANSMISSOR_PRESSAO and inst.limite_critico_alto is not None:
        return inst.valor_atual > inst.limite_critico_alto
    return False

def is_sobretemperatura(inst: Instrumento) -> bool:
    if inst.tipo == TipoInstrumento.TRANSMISSOR_TEMPERATURA and inst.limite_critico_alto is not None:
        return inst.valor_atual > inst.limite_critico_alto
    return False

def is_valvula_aberta(inst: Instrumento) -> bool:
    if inst.tipo == TipoInstrumento.VALVULA_BLOQUEIO:
        return inst.fim_de_curso_aberto is True
    return False

def is_bomba_operando(inst: Instrumento) -> bool:
    if inst.tipo == TipoInstrumento.BOMBA_MOTOR:
        return inst.motor_ligado is True
    return False

print("Predicados de instrumentacao compilados.")

## 5. Implementação do Módulo de Varredura Global (`SCADAScanningEngine`)

O motor executa a cada ciclo de scan 4 avaliações baseadas em quantificadores:
1. **Prontidão de Instrumentação (Global):** $\forall x \in \mathcal{I}, \text{IsSaudavel}(x)$
2. **Detecção de Gás Tóxico (Segurança de Área):** $\exists x \in \mathcal{D}_{gás}, \text{IsVazamentoGas}(x)$
3. **Integridade Térmica dos Reatores:** $\forall x \in \mathcal{T}_{Reator}, \neg \text{IsSobretemperatura}(x)$
4. **Permissivo de Rota de Transferência de Slurry:** $(\forall v \in \mathcal{V}_{Rota}, \text{Aberta}(v)) \land (\exists p \in \mathcal{P}_{Rota}, \text{Ligada}(p)) \land (\neg \exists s \in \mathcal{P}_{Rota}, \text{Sobrepressao}(s))$

In [ ]:
class SCADAScanningEngine:
    def __init__(self, instrumentos: List[Instrumento]):
        self.instrumentos = instrumentos

    def executar_varredura(self) -> Dict[str, Any]:
        # Regra 1: Prontidao Global de Instrumentacao
        # forall x in I, IsSaudavel(x)
        status_prontidao, falhas_inst = forall(self.instrumentos, is_saudavel)

        # Regra 2: Monitoramento Global de Gas Toxico (NH3)
        # exists x in DetectoresGas, IsVazamentoGas(x)
        guarda_gas = lambda i: i.tipo == TipoInstrumento.DETECTOR_GAS
        vazamento_detectado, testemunhas_gas = exists_in(self.instrumentos, guarda_gas, is_vazamento_gas)

        # Regra 3: Alivio Termico do Reator (Setor 100)
        # exists x in Termometros_Setor100, IsSobretemperatura(x)
        guarda_temp_s100 = lambda i: i.setor == Setor.SETOR_100 and i.tipo == TipoInstrumento.TRANSMISSOR_TEMPERATURA
        sobretemp_reator, testemunhas_temp = exists_in(self.instrumentos, guarda_temp_s100, is_sobretemperatura)

        # Regra 4: Permissivo de Linha de Transferencia (Valvulas abertas e Bomba ativa)
        valvulas_rota = lambda i: i.tag in ["XV-101", "XV-102", "XV-201"]
        bombas_rota = lambda i: i.tag in ["P-101", "M-201"]
        pressoes_rota = lambda i: i.tag in ["PT-101", "PT-201"]

        todas_valvulas_abertas, valv_fechadas = forall_in(self.instrumentos, valvulas_rota, is_valvula_aberta)
        alguma_bomba_ligada, bombas_ativas = exists_in(self.instrumentos, bombas_rota, is_bomba_operando)
        sem_sobrepressao, press_altas = forall_in(self.instrumentos, pressoes_rota, lambda i: not is_sobrepressao(i))

        permissivo_rota = todas_valvulas_abertas and alguma_bomba_ligada and sem_sobrepressao

        # Decisao de Trip Global
        trip_emergencia = vazamento_detectado or sobretemp_reator

        return {
            "Prontidao_Instrumentos": status_prontidao,
            "Falhas_Instrumentos": [i.tag for i in falhas_inst],
            "Vazamento_Gas_NH3": vazamento_detectado,
            "Detectores_Ativados": [i.tag for i in testemunhas_gas],
            "Sobretemperatura_Reator": sobretemp_reator,
            "Termometros_Alarme": [i.tag for i in testemunhas_temp],
            "Permissivo_Transferencia": permissivo_rota,
            "Valvulas_Bloqueando": [i.tag for i in valv_fechadas],
            "Trip_Emergencia_Ativo": trip_emergencia
        }

engine = SCADAScanningEngine(instrumentos_base)
res_nominal = engine.executar_varredura()
exibir_tabela(res_nominal, "Estado Nominal da Planta")

## 6. Prova Computacional das Leis de De Morgan para Quantificadores

Validamos formalmente que a negação do universal equivale ao existencial com predicado negado, e vice-versa:
$$\neg (\forall x P(x)) \equiv \exists x (\neg P(x))$$
$$\neg (\exists x Q(x)) \equiv \forall x (\neg Q(x))$$

In [ ]:
def validar_de_morgan_quantificadores(dominio: List[Instrumento]):
    # Teste 1: not(forall x Saudavel(x)) == exists x (not Saudavel(x))
    univ_saudavel, _ = forall(dominio, is_saudavel)
    lado_esq_1 = not univ_saudavel
    lado_dir_1, testemunhas_defeito = exists(dominio, lambda i: not is_saudavel(i))
    assert lado_esq_1 == lado_dir_1, "Falha na Lei de De Morgan 1"

    # Teste 2: not(exists x Vazamento(x)) == forall x (not Vazamento(x))
    guarda_gas = lambda i: i.tipo == TipoInstrumento.DETECTOR_GAS
    detectores = [i for i in dominio if guarda_gas(i)]
    
    existe_vazamento, _ = exists(detectores, is_vazamento_gas)
    lado_esq_2 = not existe_vazamento
    lado_dir_2, _ = forall(detectores, lambda i: not is_vazamento_gas(i))
    assert lado_esq_2 == lado_dir_2, "Falha na Lei de De Morgan 2"

    print("===========================================================")
    print("[OK] PROVA FORMAL: Leis de De Morgan verificadas com sucesso!")
    print(f"  1. not(forall x Saudavel(x)) [{lado_esq_1}] == exists x not(Saudavel(x)) [{lado_dir_1}]")
    print(f"  2. not(exists x Vazamento(x)) [{lado_esq_2}] == forall x not(Vazamento(x)) [{lado_dir_2}]")
    print("===========================================================")

validar_de_morgan_quantificadores(instrumentos_base)

## 7. Simulação de Cenários de Falha e Diagnóstico da Planta

Injetamos 3 cenários de contingência para testar a resposta do motor de varredura:
1. **Cenário A:** Sensor Offline (Perda de comunicação com `PT-101`).
2. **Cenário B:** Vazamento de Amônia $\text{NH}_3$ no Setor 100 (`AT-101` = $38.5\text{ ppm} > 25.0\text{ ppm}$).
3. **Cenário C:** Bloqueio de Rota por Válvula Fechada (`XV-201` Fechada).

In [ ]:
cenarios = {}

# Cenario Nominal
cenarios["0. Nominal"] = SCADAScanningEngine(criar_parque_instrumentos()).executar_varredura()

# Cenario A: Falha de Comunicacao
inst_cen_a = criar_parque_instrumentos()
inst_cen_a[0].online = False  # PT-101 offline
cenarios["1. Sensor Offline"] = SCADAScanningEngine(inst_cen_a).executar_varredura()

# Cenario B: Vazamento de Gas NH3
inst_cen_b = criar_parque_instrumentos()
inst_cen_b[3].valor_atual = 38.5  # AT-101 detecta 38.5 ppm (limite 25.0)
cenarios["2. Vazamento NH3"] = SCADAScanningEngine(inst_cen_b).executar_varredura()

# Cenario C: Bloqueio de Valvula na Rota
inst_cen_c = criar_parque_instrumentos()
inst_cen_c[11].fim_de_curso_aberto = False  # XV-201 Fechada
cenarios["3. Valvula Bloqueada"] = SCADAScanningEngine(inst_cen_c).executar_varredura()

if HAS_PANDAS:
    df_cenarios = pd.DataFrame(cenarios)
    display(df_cenarios) if 'display' in globals() else print(df_cenarios)
else:
    for nome, res in cenarios.items():
        exibir_tabela(res, nome)

## 8. Benchmark de Desempenho: Varredura com Curto-Circuito vs. Varredura Completa

Avaliamos o tempo de execução do motor de varredura para $20.000$ ciclos em uma rede expandida de $200$ instrumentos.

In [ ]:
# Gerador de parque expandido com 200 instrumentos
parque_expandido = []
for i in range(200):
    parque_expandido.append(Instrumento(
        tag=f"TT-{1000+i}",
        tipo=TipoInstrumento.TRANSMISSOR_TEMPERATURA,
        setor=Setor.SETOR_100 if i % 2 == 0 else Setor.SETOR_200,
        descricao=f"Sensor termico distribuido #{i}",
        valor_atual=random.uniform(50.0, 180.0),
        limite_critico_alto=195.0
    ))

# Funcao com curto-circuito rapido (como na logica formal standard)
def forall_short_circuit(dominio, pred):
    for x in dominio:
        if not pred(x):
            return False
    return True

# Funcao ingenua sem curto-circuito
def forall_naive(dominio, pred):
    return all([pred(x) for x in dominio])

# Inserimos uma falha no inicio para evidenciar o beneficio do curto-circuito
parque_expandido[2].online = False

N_CICLOS = 20000

t0 = time.time()
for _ in range(N_CICLOS):
    _ = forall_naive(parque_expandido, is_online)
t_naive = time.time() - t0

t0 = time.time()
for _ in range(N_CICLOS):
    _ = forall_short_circuit(parque_expandido, is_online)
t_sc = time.time() - t0

resultados_bench = [
    {"Algoritmo": "Varredura Ingenua (List Comprehension Completa)", "Tempo Total (s)": f"{t_naive:.4f}", "Tempo Medio/Scan (us)": f"{(t_naive/N_CICLOS)*1e6:.2f}"},
    {"Algoritmo": "Varredura com Curto-Circuito (LPO Otimizada)", "Tempo Total (s)": f"{t_sc:.4f}", "Tempo Medio/Scan (us)": f"{(t_sc/N_CICLOS)*1e6:.2f}"}
]
exibir_tabela(resultados_bench, "Benchmark de Desempenho do Scan")

## 9. Conclusões e Próximos Passos

1. **Poder Expressivo da Lógica de Predicados:** A quantificação universal ($\forall$) e existencial ($\exists$) permite supervisionar redes complexas com regras declarativas limpas e independentes da quantidade de sensores.
2. **Dualidade de De Morgan:** Comprovamos formal e computacionalmente que a detecção de anomalias ($\exists x \neg P(x)$) é o complemento exato da garantia de prontidão universal ($\neg \forall x P(x)$).
3. **Eficiência no Scan Cycle:** A avaliação por curto-circuito reproduz a semântica matemática e assegura latências inferiores a microssegundos por ciclo de varredura.

**Próxima Aula (Aula 07):** Validade e Inferência Lógica — Prova formal de consistência da matriz de segurança do processo.